#Reading cust from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Cleaning customer Info

In [0]:
%python 
querry = """
with cte as (
select cst_id as id ,
row_number()over(partition by cst_id order by cst_create_date desc) as rn ,
cst_key as key,trim(cst_firstname) as firstname,trim(cst_lastname) as lastname,
case when cst_marital_status = 'M' then 'Married' else 'Single'
end as marital_status,
case when cst_gndr = 'M' then 'Male' else 'Female'
end as gndr,
cst_create_date as create_date 
from workspace.bronze.crm_cust_info
where cst_id IS NOT null )
select id , key , firstname,lastname,marital_status,gndr,create_date from cte
where rn = 1"""

df = spark.sql(querry)
df.write.mode("overwrite").\
    format("delta").\
    option("overwriteSchema", "true").\
      saveAsTable("workspace.silver.crm_cust_info")



#DQ Summary for Customer INFO

In [0]:
%python
querry="""
WITH customer_dq AS (
    SELECT
        cst_id,
        cst_gndr,
        COUNT(*) OVER (
            PARTITION BY cst_id
        ) AS duplicate_count
    FROM workspace.bronze.crm_cust_info
)

SELECT
    COUNT(*) AS Total_Records,

    SUM(CASE
        WHEN cst_id IS NULL THEN 1
        ELSE 0
    END) AS Missing_IDs,

    SUM(CASE
        WHEN cst_id IS NOT NULL
             AND duplicate_count > 1 THEN 1
        ELSE 0
    END) AS Duplicate_Records,

    SUM(CASE
        WHEN cst_gndr IS NULL
             OR TRIM(cst_gndr) = '' THEN 1
        ELSE 0
    END) AS Missing_Gender

FROM customer_dq"""

df = spark.sql(querry)
display(df)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.customer_dq_summary")

#Reading Product Data from Bronze

In [0]:
%python
df= spark.read.table("workspace.bronze.crm_prd_info")

#CleanUP Product table

In [0]:
%python 
query = """
with prd_table as (
select prd_id as id ,
prd_key as Product_key,
prd_nm as Product_name,prd_cost as cost,
case when prd_cost <=0 then 'Invalid' 
     when prd_cost > 1 then  'Valid'
     else 'Missing' end as Cost_status,
prd_line as line ,
case when UPPER(TRIM(prd_line)) = 'M' then 'Mountain'
        when UPPER(TRIM(prd_line)) = 'R' then 'Road'
        when UPPER(TRIM(prd_line)) = 'T' then 'Touring'
        when UPPER(TRIM(prd_line)) = 'S' then 'Other Sales'
     else 'Unknown' end as product_category,
prd_start_dt as start_date,
prd_end_dt as end_date,
case  when prd_start_dt is null then 'Missing Start Date' 
        when date(prd_end_dt)  IS NULL then 'Active' 
        when date(prd_end_dt) < date(prd_start_dt) then 'Invalid'
        else 'Valid' 
end as date_status
from workspace.bronze.crm_prd_info)
select * from prd_table"""

df=spark.sql(query)
display(df)
df.write.mode("overwrite").\
format("delta").\
option("overwriteSchema", "true").\
saveAsTable("silver.tbl_prd_info")



#Read sales details from Bronze

In [0]:
df=spark.read.table("workspace.bronze.sales_details")
df.display()


#Clean sales Details Table

In [0]:
%python
query = """
with cte as (
select sls_ord_num as order_number,
sls_prd_key as product_key,
sls_cust_id as customer_id,
TRY_TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd') AS order_date,
TRY_TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') AS ship_date,
TRY_TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd') AS due_date,
case when TRY_TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') < TRY_TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd') or TRY_TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd') < TRY_TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd') then 'Invalid'
else 'Valid'
end as date_status,
sls_sales as sales,
sls_quantity as quantity,
sls_price as price
from workspace.bronze.sales_details) 
,cte2 as (
select order_number,product_key,customer_id,order_date,ship_date,due_date,date_status,sales,quantity,price,
case when sales is null then 'Missing'
     when price is NULL then 'Missing'
     when quantity is NULL then 'Missing'
     when sales < expected_sale then 'Invalid'
     when sales > expected_sale then 'Invalid'
     when sales = expected_sale then 'Valid'
end as sales_status
from (
        select order_number,product_key,customer_id,order_date,ship_date,due_date,date_status,sales,quantity,price,
        quantity*price as expected_sale
        from cte ) as t ) 
select * from cte2"""

df= spark.sql(query)
df.write.mode("overwrite").\
format("delta").\
option("overwriteSchema", "true").\
saveAsTable("silver.tbl_sales_details")
df.display()